In [4]:
import requests
import pandas as pd
import sqlite3

url = "https://restcountries.com/v3.1/all?fields=name,capital,region,subregion,population,area,currencies"
response = requests.get(url)

if response.status_code != 200:
    print(f"Błąd API! Kod statusu: {response.status_code}")
    print("Treść odpowiedzi:", response.text)
else:
    data = response.json()
    
    if isinstance(data, dict):
         print("Uwaga: API zwróciło słownik (prawdopodobnie komunikat o błędzie) zamiast listy krajów.")
         print("Zwrócone dane:", data)
    else:
        kraje_dane = []

        for kraj in data:
            nazwa = kraj.get("name", {}).get("common", None)
            
            stolica = ", ".join(kraj.get("capital", [])) if "capital" in kraj else None
            
            region = kraj.get("region", None)
            subregion = kraj.get("subregion", None)
            
            populacja = kraj.get("population", None)
            powierzchnia = kraj.get("area", None)
            
            waluty_dict = kraj.get("currencies", {})
            waluta = ", ".join(waluty_dict.keys()) if waluty_dict else None
            
            kraje_dane.append({
                "nazwa": nazwa,
                "stolica": stolica,
                "region": region,
                "subregion": subregion,
                "populacja": populacja,
                "powierzchnia": powierzchnia,
                "waluta": waluta
            })

        df_kraje = pd.DataFrame(kraje_dane)

        print("Kształt (shape):", df_kraje.shape)
        print("\nTypy danych (dtypes):\n", df_kraje.dtypes)
        print("\nPierwsze 5 wierszy (head):")
        display(df_kraje.head())

Kształt (shape): (250, 7)

Typy danych (dtypes):
 nazwa               str
stolica             str
region              str
subregion           str
populacja         int64
powierzchnia    float64
waluta              str
dtype: object

Pierwsze 5 wierszy (head):


,nazwa,stolica,region,subregion,populacja,powierzchnia,waluta
0,Anguilla,The Valley,Americas,Caribbean,16010,91.0,XCD
1,Guatemala,Guatemala City,Americas,Central America,18079810,108889.0,GTQ
2,Gambia,Banjul,Africa,Western Africa,2422712,10689.0,GMD
3,Mexico,Mexico City,Americas,North America,130575786,1964375.0,MXN
4,Malawi,Lilongwe,Africa,Eastern Africa,20734262,118484.0,MWK


In [2]:
conn = sqlite3.connect("kraje_swiata.db")

df_kraje.to_sql("kraje", conn, if_exists="replace", index=False)

print("Pomyślnie zapisano dane w bazie 'kraje_swiata.db'")

Pomyślnie zapisano dane w bazie 'kraje_swiata.db'


In [3]:
df_populacja_swiata = pd.read_sql_query("""
    SELECT SUM(populacja) AS laczna_populacja_swiata 
    FROM kraje
""", conn)

print("Łączna populacja świata:")
display(df_populacja_swiata)

df_top10_populacja = pd.read_sql_query("""
    SELECT nazwa, populacja 
    FROM kraje 
    ORDER BY populacja DESC 
    LIMIT 10
""", conn)

print("\n10 krajów z największą populacją:")
display(df_top10_populacja)

df_top10_powierzchnia = pd.read_sql_query("""
    SELECT nazwa, powierzchnia 
    FROM kraje 
    ORDER BY powierzchnia DESC 
    LIMIT 10
""", conn)

print("\n10 krajów o największej powierzchni:")
display(df_top10_powierzchnia)

conn.close()

Łączna populacja świata:


,laczna_populacja_swiata
0,8019495460



10 krajów z największą populacją:


,nazwa,populacja
0,India,1417492000
1,China,1408280000
2,United States,340110988
3,Indonesia,284438782
4,Pakistan,241499431
5,Nigeria,223800000
6,Brazil,213421037
7,Bangladesh,169828911
8,Russia,146028325
9,Mexico,130575786



10 krajów o największej powierzchni:


,nazwa,powierzchnia
0,Russia,17098246.0
1,Antarctica,14000000.0
2,Canada,9984670.0
3,China,9706961.0
4,United States,9525067.0
5,Brazil,8515767.0
6,Australia,7692024.0
7,India,3287263.0
8,Argentina,2780400.0
9,Kazakhstan,2724900.0
